# Deep Learning Recommenders — Exercises

Companion to the note [Deep Learning Recommenders](Deep%20Learning%20Recommenders.md).

Practise the building blocks of neural recommenders without a deep-learning framework: embedding-table sizing and hashing, two-tower models trained with in-batch softmax (and logQ correction), brute-force retrieval, factorization-machine interactions, LightGCN propagation, causal self-attention over item sequences (SASRec) and the linear autoencoder EASE, a strong baseline. Everything runs in NumPy on small synthetic interaction data.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax, log_softmax, logsumexp, expit
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import Ridge
rng = np.random.default_rng(0)

# Synthetic two-tower data: users and items with side features, interactions from a hidden bilinear taste model
_g = np.random.default_rng(0)
NU, NI, DU, DI = 300, 200, 6, 5
Xu = _g.normal(size=(NU, DU))            # user features
Yi = _g.normal(size=(NI, DI))            # item features
_logits = Xu @ _g.normal(size=(DU, DI)) @ Yi.T
train_pairs, test_item = [], []
for _u in range(NU):
    _items = _g.choice(NI, 6, replace=False, p=softmax(_logits[_u]))
    train_pairs += [(_u, _i) for _i in _items[:5]]
    test_item.append(_items[5])
train_pairs = np.array(train_pairs); test_item = np.array(test_item)

# Synthetic implicit matrix for EASE: 200 users x 60 items, one held-out positive per user
_g = np.random.default_rng(1)
X_ease = ((_g.normal(size=(200, 4)) @ _g.normal(size=(60, 4)).T + 0.7 * _g.normal(size=(200, 60))
           + np.linspace(1, -1, 60)) > 1.5).astype(float)
X_ease_tr = X_ease.copy(); ease_test = np.full(200, -1)
for _u in range(200):
    _p = np.flatnonzero(X_ease[_u])
    if len(_p) >= 2:
        ease_test[_u] = _g.choice(_p); X_ease_tr[_u, ease_test[_u]] = 0

## Part A · Concepts

### Exercise 1 · Memorisation vs generalisation
*🧠 Concept · ★☆☆*

Wide & Deep combines a **wide** linear model on (crossed) sparse features with a **deep** MLP on embeddings.
Explain with an app-store example what "memorisation" and "generalisation" mean, which part does which, and what failure mode each part alone has.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of a crossed feature like `installed=Netflix AND impression=Hulu` vs embeddings of 'video streaming apps'.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Wide (memorisation):** learns weights for specific, frequently co-occurring feature crosses such as `installed=Netflix × shown=Hulu`.
  It is precise for patterns seen often, but cannot generalise to crosses never seen in training. It also needs manual feature engineering.
- **Deep (generalisation):** embeds users/apps in a dense space, so "streaming apps" end up close and the model can score unseen combinations.
  Alone it can **over-generalise**: with sparse, high-rank interactions it recommends loosely related items, even to niche users.

Trained jointly, the wide part corrects the deep part on known exceptions while the deep part covers the long tail.

</details>

### Exercise 2 · MLP vs dot product
*🧠 Concept · ★★☆*

NCF replaces the dot product $p_u^\top q_i$ with an MLP $f([p_u; q_i])$. Rendle et al. (2020) found well-tuned MF matches or beats NCF.
Give two reasons why, and one **systems** reason why retrieval models keep the dot product.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

How many hidden units does an MLP need to represent $\sum_f a_f b_f$ accurately? And what does ANN search assume about the score?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Learnability:** an MLP is a universal approximator, but approximating the simple bilinear dot product is hard. It needs many units and
   lots of data, and with sparse data it learns it poorly. The dot product builds that inductive bias in for free.
2. **Baselines:** many papers compared against under-tuned MF. With proper regularisation, dimension and negative sampling, MF/iALS is very strong.

**Systems:** with $s(u,i)=\langle e_u, e_i\rangle$ the item embeddings can be indexed once and served by maximum inner product search
(FAISS/ScaNN/HNSW) in sub-linear time. An MLP over $(u,i)$ must be evaluated for every item, which is far too slow for retrieval.
So MLPs belong in the **ranking** stage ([[Recommender Systems Overview]]).

</details>

### Exercise 3 · What a two-tower model cannot see
*🧠 Concept · ★☆☆*

In a two-tower model the user tower and the item tower only meet in the final dot product. Name a useful feature interaction such a model
**cannot** capture, explain why this restriction is accepted, and where in the pipeline the missing interactions are modelled instead.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Can the user tower know which item it is being scored against?

</details>

<details>
<summary><b>✅ Solution</b></summary>

It cannot use **early cross features** between user and item, e.g. "has the user watched *this* item's director before?" or "price of *this* item relative to
the user's average spend". The user embedding is computed without knowing the item.

This is accepted because separability is what makes **precomputation + ANN** possible: item embeddings are computed offline, the user embedding once per
request. The rich cross features are used by the **ranking** model on the few hundred retrieved candidates (DLRM, DCN, GBDT).

</details>

### Exercise 4 · Sequential recommenders
*🧠 Concept · ★★☆*

Compare GRU4Rec, SASRec and BERT4Rec: how each one encodes the user's item sequence, its training objective, and why BERT4Rec must use a
**masked** objective while SASRec uses a **causal** mask. What leakage would occur if SASRec were trained with bidirectional attention?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

SASRec predicts the next item at every position. What could position $t$ see if attention were bidirectional?

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **GRU4Rec:** an RNN over the session. It predicts the next item and processes the sequence sequentially (slow to train).
- **SASRec:** a Transformer decoder (causal self-attention). At each position $t$ it predicts item $t+1$ from items $\le t$, and all positions train in parallel.
- **BERT4Rec:** a bidirectional Transformer encoder trained with a **cloze** task: random items are replaced by `[MASK]` and predicted from both sides.

If SASRec used bidirectional attention, position $t$ could attend to item $t+1$, the very target it must predict. The task becomes copying, and the
model learns nothing useful (label leakage). BERT4Rec avoids this by hiding the target (masking) instead of hiding the future. At inference it appends
`[MASK]` at the end. See [[Transformers]].

</details>

## Part B · By hand

### Exercise 5 · Sizing embedding tables
*✍️ By hand · ★☆☆*

A system has $10^8$ users and $10^7$ items, each with a 64-dimensional float32 embedding. How many GB (= $10^9$ bytes) do the user table and the item
table take? How much does the user table shrink with 8-bit quantisation?

In [ ]:
gb_users = None
gb_items = None
gb_users_int8 = None

check('users GB', gb_users, 1e8 * 64 * 4 / 1e9)
check('items GB', gb_items, 1e7 * 64 * 4 / 1e9)
check('users int8 GB', gb_users_int8, 1e8 * 64 / 1e9)

<details>
<summary><b>💡 Hint</b></summary>

float32 = 4 bytes, int8 = 1 byte.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Users: $10^8\cdot 64\cdot 4 = 2.56\cdot10^{10}$ B $= 25.6$ GB. Items: $2.56$ GB. int8 users: $6.4$ GB.
Embedding tables, not MLP weights, dominate model size. This is why sharding, hashing, mixed-dimension and quantised embeddings exist.

```python
gb_users = 25.6
gb_items = 2.56
gb_users_int8 = 6.4
```

</details>

### Exercise 6 · The hashing trick and collisions
*✍️ By hand · ★★☆*

Instead of one row per ID, $n = 10^6$ IDs are hashed uniformly into $m = 2^{20}$ buckets. What is the probability that a given ID **shares** its bucket
with at least one other ID? Give the exact value and the approximation $1-e^{-n/m}$. What does a collision do to the model?

In [ ]:
p_collide = None

_n, _m = 10**6, 2**20
_sim = np.random.default_rng(0).integers(0, _m, _n); _cnt = np.bincount(_sim, minlength=_m)
print('simulated fraction:', round(np.mean(_cnt[_sim] > 1), 4))
check('collision probability', p_collide, 1 - (1 - 1 / _m) ** (_n - 1), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Each of the other $n-1$ IDs misses the given bucket with probability $1-1/m$, independently.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P = 1-(1-1/m)^{n-1} \approx 1-e^{-n/m} = 1-e^{-0.9537} \approx 0.615$.
About 61% of IDs share an embedding with at least one other ID. The shared vector is pulled towards an average of unrelated entities.
This hurts most for frequent IDs. Remedies: more buckets, multiple hash functions whose embeddings are summed (Bloom/"QR" embeddings), or full tables for head IDs.

```python
p_collide = 1 - (1 - 2**-20) ** (10**6 - 1)
```

</details>

### Exercise 7 · In-batch softmax by hand
*✍️ By hand · ★★☆*

In a batch, a user's scores against the batch items are $s = (2.0,\ 1.0,\ 0.5)$, where the first item is the positive.
Compute the loss $L=-\log\frac{e^{s^+/\tau}}{\sum_j e^{s_j/\tau}}$ for $\tau=1$ and for $\tau=0.5$. What does a lower temperature do?

In [ ]:
loss_tau1 = None
loss_tau05 = None

_s = np.array([2, 1, .5])
check('tau = 1', loss_tau1, -log_softmax(_s)[0], tol=1e-3)
check('tau = 0.5', loss_tau05, -log_softmax(_s / .5)[0], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\tau=1$: $L = -2 + \log(e^2+e^1+e^{0.5})$; for $\tau=0.5$ double all scores first.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\tau=1$: $\log(7.389+2.718+1.649)=\log 11.756 = 2.464$, so $L \approx 0.464$.
$\tau=0.5$: scores $(4,2,1)$, $\log(54.60+7.389+2.718) = \log 64.71 = 4.170$, so $L\approx 0.170$.

A lower $\tau$ sharpens the softmax. The loss concentrates on the **hardest** negatives (those with scores close to the positive), and
easy ones barely contribute. Too low a $\tau$ makes training unstable and over-punishes false negatives.

```python
loss_tau1 = -2 + np.log(np.exp(2) + np.exp(1) + np.exp(0.5))
loss_tau05 = -4 + np.log(np.exp(4) + np.exp(2) + np.exp(1))
```

</details>

### Exercise 8 · logQ correction
*✍️ By hand · ★★☆*

In-batch negatives are sampled proportionally to item popularity, so popular items appear as negatives far more often than under uniform sampling.
The **logQ correction** replaces $s_j$ by $s_j - \log q_j$, where $q_j$ is the probability that item $j$ appears in a batch.

With the scores of Exercise 7 ($\tau=1$) and $q = (0.01,\ 0.2,\ 0.05)$, compute the corrected scores and the corrected loss.
Why does the correction help niche items?

In [ ]:
s_corr = None   # 3 numbers
loss_corr = None

_s = np.array([2, 1, .5]) - np.log([.01, .2, .05])
check('corrected scores', s_corr, _s, tol=1e-3)
check('corrected loss', loss_corr, -log_softmax(_s)[0], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$-\log 0.01 = 4.605$, $-\log 0.2 = 1.609$, $-\log 0.05 = 2.996$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Corrected scores $\approx (6.605,\ 2.609,\ 3.496)$, loss $= -6.605 + \log(e^{6.605}+e^{2.609}+e^{3.496}) \approx 0.061$.

Without correction, a popular item is pushed down every time it shows up as a negative, so the model learns "popular = bad" for most users,
and a rare positive gets too little credit. Subtracting $\log q_j$ makes the sampled softmax an unbiased estimate of the full softmax
(sampled softmax theory, Yi et al. 2019). Rare items (small $q$) get a big boost and popular ones a small one, which undoes the sampling bias.

```python
s_corr = list(np.array([2, 1, .5]) - np.log([.01, .2, .05]))
loss_corr = -s_corr[0] + logsumexp(s_corr)
```

</details>

### Exercise 9 · One LightGCN layer by hand
*✍️ By hand · ★★☆*

Bipartite graph: user $u_0$ interacted with items $i_0, i_1$; user $u_1$ with $i_1, i_2$. LightGCN propagates (no weights, no non-linearity)
$$e_u^{(k+1)} = \sum_{i\in N(u)}\frac{1}{\sqrt{|N(u)|}\sqrt{|N(i)|}}\,e_i^{(k)}.$$
With 1-dimensional item embeddings $e_{i_0}=1$, $e_{i_1}=2$, $e_{i_2}=-1$, compute $e_{u_0}^{(1)}$ and $e_{u_1}^{(1)}$.

In [ ]:
e_u0 = None
e_u1 = None

_X = np.array([[1, 1, 0], [0, 1, 1.]]); _du = _X.sum(1); _di = _X.sum(0)
_E = (_X / np.sqrt(np.outer(_du, _di))) @ np.array([1, 2, -1.])
check('e_u0', e_u0, _E[0])
check('e_u1', e_u1, _E[1])

<details>
<summary><b>💡 Hint</b></summary>

Degrees: $|N(u_0)|=|N(u_1)|=2$, $|N(i_0)|=|N(i_2)|=1$, $|N(i_1)|=2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$e_{u_0}^{(1)} = \frac{1}{\sqrt2\sqrt1}\cdot1 + \frac1{\sqrt2\sqrt2}\cdot2 = 0.7071 + 1 = 1.7071$.
$e_{u_1}^{(1)} = \frac12\cdot2 + \frac1{\sqrt2}\cdot(-1) = 1 - 0.7071 = 0.2929$.
The symmetric normalisation down-weights high-degree (popular) neighbours, so the niche items $i_0$, $i_2$ count more per edge than the shared item $i_1$.

```python
e_u0 = 1 / np.sqrt(2) + 1
e_u1 = 1 - 1 / np.sqrt(2)
```

</details>

### Exercise 10 · Counting attention pairs
*✍️ By hand · ★☆☆*

A sequence of $T=4$ items goes through one self-attention layer. How many (query, key) pairs are allowed (a) with SASRec's causal mask and
(b) with BERT4Rec's bidirectional attention? Write down the causal mask matrix (1 = allowed).

In [ ]:
pairs_causal = None
pairs_bidir = None

check('causal', pairs_causal, np.tril(np.ones((4, 4))).sum())
check('bidirectional', pairs_bidir, 16)

<details>
<summary><b>💡 Hint</b></summary>

Position $t$ (0-based) may attend to positions $0,\dots,t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Causal: $1+2+3+4 = T(T+1)/2 = 10$ pairs; mask $= \begin{pmatrix}1&0&0&0\\1&1&0&0\\1&1&1&0\\1&1&1&1\end{pmatrix}$ (lower triangular).
Bidirectional: $T^2 = 16$. In code the masked entries get score $-\infty$ before the softmax.

```python
pairs_causal = 10
pairs_bidir = 16
```

</details>

## Part C · Code from scratch

### Exercise 11 · In-batch softmax loss and its gradient
*💻 Code · ★★☆*

For a batch of $B$ user embeddings `U` ($B\times d$) and their positive item embeddings `V` ($B\times d$, row $b$ is the positive of user $b$; the other
rows act as negatives), implement
$$L = -\frac1B\sum_b \log \mathrm{softmax}_b\big(UV^\top/\tau\big)_{bb}$$
and its gradients $\partial L/\partial U = \frac{1}{\tau B}(P - I)V$ and $\partial L/\partial V = \frac{1}{\tau B}(P-I)^\top U$, where $P$ is the row-wise softmax.

In [ ]:
def inbatch_loss_grad(U, V, tau):
    # TODO: return loss, dU, dV
    return None

_g = np.random.default_rng(1)
U_b, V_b = _g.normal(size=(4, 3)), _g.normal(size=(4, 3))
out_ib = inbatch_loss_grad(U_b, V_b, 0.5)

_ref_loss = lambda U, V: -np.mean(np.diag(log_softmax(U @ V.T / 0.5, axis=1)))
if out_ib is not None:
    check('loss', out_ib[0], _ref_loss(U_b, V_b))
    _eps = 1e-6; _num = np.zeros_like(U_b)
    for a in range(4):
        for c in range(3):
            E = np.zeros_like(U_b); E[a, c] = _eps
            _num[a, c] = (_ref_loss(U_b + E, V_b) - _ref_loss(U_b - E, V_b)) / (2 * _eps)
    check('dL/dU (finite differences)', out_ib[1], _num, tol=1e-5)
    _numV = np.zeros_like(V_b)
    for a in range(4):
        for c in range(3):
            E = np.zeros_like(V_b); E[a, c] = _eps
            _numV[a, c] = (_ref_loss(U_b, V_b + E) - _ref_loss(U_b, V_b - E)) / (2 * _eps)
    check('dL/dV (finite differences)', out_ib[2], _numV, tol=1e-5)
else:
    check('in-batch loss', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Use `scipy.special.log_softmax(S, axis=1)` with `S = U @ V.T / tau` for a numerically stable loss.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

The gradient of the cross-entropy w.r.t. the logits is `(P - I) / B`; then chain through $S = UV^\top/\tau$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is cross-entropy where the "class" of user $b$ is its own positive among the $B$ items in the batch, so with $B$ users you get $B-1$ negatives for free.
Caveat: if the same item appears twice in a batch, one copy is a **false negative** for the other user. Real implementations mask such duplicates.

```python
def inbatch_loss_grad(U, V, tau):
    B = len(U)
    S = U @ V.T / tau
    loss = -np.mean(np.diag(log_softmax(S, axis=1)))
    G = (softmax(S, axis=1) - np.eye(B)) / (tau * B)
    return loss, G @ V, G.T @ U

_g = np.random.default_rng(1)
U_b, V_b = _g.normal(size=(4, 3)), _g.normal(size=(4, 3))
out_ib = inbatch_loss_grad(U_b, V_b, 0.5)
```

</details>

### Exercise 12 · Training a two-tower model
*💻 Code · ★★★*

Train linear towers $e_u = x_u W_u$, $e_i = y_i W_i$ ($W_u\in\mathbb R^{6\times8}$, $W_i\in\mathbb R^{5\times8}$) on `train_pairs` (user features `Xu`,
item features `Yi`) with the in-batch softmax of the previous exercise: batch size 64, $\tau=0.5$, learning rate 0.05, 1500 SGD steps,
init $\mathcal N(0, 0.1^2)$. Chain rule: $\partial L/\partial W_u = X_b^\top\,\partial L/\partial U$.

Then implement `recall_at_k(Wu, Wi, K)`: score all items for every user, exclude training items, and return the fraction of users whose
`test_item` is in the top-K. Store recall@10 after training in `recall_tt`.

In [ ]:
def recall_at_k(Wu, Wi, K=10):
    return None

def train_two_tower(steps=1500, B=64, tau=0.5, lr=0.05, d=8, seed=0):
    # TODO: return Wu, Wi
    return None

recall_tt = None

_pop = np.bincount(train_pairs[:, 1], minlength=NI)
_S = np.tile(_pop.astype(float), (NU, 1)); _S[train_pairs[:, 0], train_pairs[:, 1]] = -np.inf
_rpop = np.mean([test_item[u] in np.argsort(-_S[u])[:10] for u in range(NU)])
print('popularity recall@10:', round(_rpop, 3))
if recall_tt is not None:
    _W = train_two_tower()
    _S2 = (Xu @ _W[0]) @ (Yi @ _W[1]).T; _S2[train_pairs[:, 0], train_pairs[:, 1]] = -np.inf
    check('your recall_at_k', recall_at_k(*_W, 10), np.mean([test_item[u] in np.argsort(-_S2[u])[:10] for u in range(NU)]))
    check('two-tower far beats popularity', recall_tt > 3 * _rpop, True)
else:
    check('two-tower recall', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Each step: `b = train_pairs[g.choice(len(train_pairs), B, replace=False)]`, `U = Xu[b[:,0]] @ Wu`, `V = Yi[b[:,1]] @ Wi`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`_, dU, dV = inbatch_loss_grad(U, V, tau)`; `Wu -= lr * Xu[b[:,0]].T @ dU`; `Wi -= lr * Yi[b[:,1]].T @ dV`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Recall@10 rises from ≈0.10 (random init) to ≈0.69, versus ≈0.18 for popularity. The towers use only **features**, so the same model also scores
brand-new users and items that have features (no cold start), unlike ID-based MF. The data was generated by a bilinear model $x_u^\top A y_i$,
which linear towers can represent as $W_uW_i^\top\approx A$ (rank ≤ 8).

```python
def recall_at_k(Wu, Wi, K=10):
    S = (Xu @ Wu) @ (Yi @ Wi).T
    S[train_pairs[:, 0], train_pairs[:, 1]] = -np.inf
    top = np.argsort(-S, axis=1)[:, :K]
    return float(np.mean([test_item[u] in top[u] for u in range(NU)]))

def train_two_tower(steps=1500, B=64, tau=0.5, lr=0.05, d=8, seed=0):
    g = np.random.default_rng(seed)
    Wu = 0.1 * g.normal(size=(DU, d)); Wi = 0.1 * g.normal(size=(DI, d))
    for _ in range(steps):
        b = train_pairs[g.choice(len(train_pairs), B, replace=False)]
        U = Xu[b[:, 0]] @ Wu; V = Yi[b[:, 1]] @ Wi
        _, dU, dV = inbatch_loss_grad(U, V, tau)
        Wu -= lr * Xu[b[:, 0]].T @ dU
        Wi -= lr * Yi[b[:, 1]].T @ dV
    return Wu, Wi

Wu_tt, Wi_tt = train_two_tower()
recall_tt = recall_at_k(Wu_tt, Wi_tt, 10)
print('two-tower recall@10:', round(recall_tt, 3))
```

</details>

### Exercise 13 · Exact retrieval vs a nearest-neighbour index
*💻 Code · ★☆☆*

Implement `retrieve(user_vecs, item_vecs, K)` returning, for each user, the indices of the K items with the highest inner product (brute force,
one matrix product + `argpartition`/`argsort`). Use L2-normalised random embeddings, for which inner product = cosine, and compare with
`sklearn.neighbors.NearestNeighbors(metric='cosine')`. Why do production systems use *approximate* NN instead?

In [ ]:
_g = np.random.default_rng(2)
users_e = _g.normal(size=(5, 16)); items_e = _g.normal(size=(1000, 16))
users_e /= np.linalg.norm(users_e, axis=1, keepdims=True); items_e /= np.linalg.norm(items_e, axis=1, keepdims=True)

def retrieve(user_vecs, item_vecs, K):
    return None

top_idx = retrieve(users_e, items_e, 5)

_nn = NearestNeighbors(n_neighbors=5, metric='cosine').fit(items_e)
check('top-5 neighbours', top_idx, _nn.kneighbors(users_e, return_distance=False))

<details>
<summary><b>💡 Hint</b></summary>

`S = user_vecs @ item_vecs.T`; `np.argsort(-S, axis=1)[:, :K]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Brute force costs $O(N_{items}\cdot d)$ per query. At $10^8$ items and thousands of queries per second that is too much, so ANN indexes (HNSW graphs,
IVF-PQ in FAISS, ScaNN) trade a little recall for orders of magnitude speed. For un-normalised embeddings, maximum inner product search (MIPS) is
**not** the same as cosine/L2 NN. Indexes either support MIPS directly or append an extra dimension to reduce MIPS to L2 search.

```python
_g = np.random.default_rng(2)
users_e = _g.normal(size=(5, 16)); items_e = _g.normal(size=(1000, 16))
users_e /= np.linalg.norm(users_e, axis=1, keepdims=True); items_e /= np.linalg.norm(items_e, axis=1, keepdims=True)

def retrieve(user_vecs, item_vecs, K):
    S = user_vecs @ item_vecs.T
    return np.argsort(-S, axis=1)[:, :K]

top_idx = retrieve(users_e, items_e, 5)
```

</details>

### Exercise 14 · Factorization-machine interactions in O(kn)
*💻 Code · ★★☆*

Implement `fm_pairwise(x, V)` computing $\sum_{i<j}\langle v_i,v_j\rangle x_ix_j$ with the $O(kn)$ identity
$\frac12\sum_f[(\sum_i v_{if}x_i)^2 - \sum_i v_{if}^2x_i^2]$, **vectorised over a batch** of rows `x` (shape $N\times n$).
This is the interaction layer of DeepFM ([[Collaborative Filtering and Matrix Factorization]], Exercise 12).

In [ ]:
_g = np.random.default_rng(3)
x_fm = (_g.random((6, 10)) < 0.3) * _g.normal(size=(6, 10))
V_fm = _g.normal(size=(10, 4))

def fm_pairwise(x, V):
    return None

fm_out = fm_pairwise(x_fm, V_fm)

_ref = [sum(V_fm[i] @ V_fm[j] * r[i] * r[j] for i in range(10) for j in range(i + 1, 10)) for r in x_fm]
check('pairwise interactions (vs brute force O(n^2))', fm_out, _ref)

<details>
<summary><b>💡 Hint</b></summary>

`xv = x @ V` (N×k) and `x2v2 = (x**2) @ (V**2)`; result `0.5 * (xv**2 - x2v2).sum(1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Two matrix products replace the double loop. DeepFM feeds the same embeddings $V$ to the FM component and to an MLP. DCN instead learns explicit
higher-order crosses $x_{l+1} = x_0 (x_l^\top w_l) + b_l + x_l$.

```python
_g = np.random.default_rng(3)
x_fm = (_g.random((6, 10)) < 0.3) * _g.normal(size=(6, 10))
V_fm = _g.normal(size=(10, 4))

def fm_pairwise(x, V):
    xv = x @ V
    x2v2 = (x ** 2) @ (V ** 2)
    return 0.5 * (xv ** 2 - x2v2).sum(1)

fm_out = fm_pairwise(x_fm, V_fm)
```

</details>

### Exercise 15 · LightGCN propagation
*💻 Code · ★★☆*

Implement `lightgcn(X, E_user, E_item, K)` for an interaction matrix `X` (users × items):
build the normalised bipartite adjacency $\tilde A = D^{-1/2}AD^{-1/2}$ with $A = \begin{pmatrix}0&X\\X^\top&0\end{pmatrix}$, propagate
$E^{(k+1)} = \tilde A E^{(k)}$ for $K$ layers and return the **mean** of $E^{(0)},\dots,E^{(K)}$, split back into users and items.
Test on the 2×3 graph from Exercise 9 (with 2-dimensional embeddings) and on a random 20×15 graph.

In [ ]:
def lightgcn(X, E_user, E_item, K=2):
    # TODO: return (final_user_emb, final_item_emb)
    return None

_g = np.random.default_rng(4)
X_lg = (_g.random((20, 15)) < 0.2).astype(float); X_lg[0, 0] = X_lg[1, 1] = 1
Eu0, Ei0 = _g.normal(size=(20, 2)), _g.normal(size=(15, 2))
lg_out = lightgcn(X_lg, Eu0, Ei0, K=3)

def _ref_lightgcn(X, Eu, Ei, K):   # edge-by-edge message passing
    du, di = X.sum(1), X.sum(0)
    layers_u, layers_i = [Eu], [Ei]
    for _ in range(K):
        nu_ = np.zeros_like(Eu); ni_ = np.zeros_like(Ei)
        for u, i in zip(*np.nonzero(X)):
            w = 1 / np.sqrt(du[u] * di[i])
            nu_[u] += w * layers_i[-1][i]; ni_[i] += w * layers_u[-1][u]
        layers_u.append(nu_); layers_i.append(ni_)
    return np.mean(layers_u, 0), np.mean(layers_i, 0)
if lg_out is not None:
    _r = _ref_lightgcn(X_lg, Eu0, Ei0, 3)
    check('user embeddings', lg_out[0], _r[0]); check('item embeddings', lg_out[1], _r[1])
    _t = lightgcn(np.array([[1, 1, 0], [0, 1, 1.]]), np.zeros((2, 1)), np.array([[1.], [2.], [-1.]]), K=1)
    check('Exercise 9 (mean of layers 0 and 1)', _t[0].ravel(), [(0 + 1.7071068) / 2, (0 + 0.2928932) / 2])
else:
    check('LightGCN', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Degrees of isolated nodes are 0. Use `dinv = np.zeros_like(deg); dinv[deg > 0] = deg[deg > 0] ** -0.5` to avoid division by zero.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`A = np.block([[np.zeros((nu, nu)), X], [X.T, np.zeros((ni, ni))]])`; `E = np.vstack([E_user, E_item])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

LightGCN drops the feature transforms and non-linearities of a GCN, since with only ID embeddings they hurt. Averaging layers mixes 0-hop (own ID),
1-hop (direct neighbours) and 2-hop (similar users / co-consumed items) signals and limits over-smoothing. Training then uses BPR on the final embeddings
([[Graph Neural Networks]]).

```python
def lightgcn(X, E_user, E_item, K=2):
    nu, ni = X.shape
    A = np.block([[np.zeros((nu, nu)), X], [X.T, np.zeros((ni, ni))]])
    deg = A.sum(1)
    dinv = np.zeros_like(deg)
    dinv[deg > 0] = deg[deg > 0] ** -0.5
    A_hat = dinv[:, None] * A * dinv[None, :]
    E = np.vstack([E_user, E_item])
    layers = [E]
    for _ in range(K):
        layers.append(A_hat @ layers[-1])
    F = np.mean(layers, axis=0)
    return F[:nu], F[nu:]

_g = np.random.default_rng(4)
X_lg = (_g.random((20, 15)) < 0.2).astype(float); X_lg[0, 0] = X_lg[1, 1] = 1
Eu0, Ei0 = _g.normal(size=(20, 2)), _g.normal(size=(15, 2))
lg_out = lightgcn(X_lg, Eu0, Ei0, K=3)
```

</details>

### Exercise 16 · Causal self-attention over an item sequence
*💻 Code · ★★☆*

Implement one SASRec-style attention head: given the embedded sequence `E` ($T\times d$) and projections `Wq, Wk, Wv` ($d\times d$),
return $\mathrm{softmax}\big(\frac{QK^\top}{\sqrt d} + M\big)V$, where $M_{ts}=-\infty$ for $s>t$.
Then verify **causality**: changing the last item must not change the outputs at earlier positions.

In [ ]:
_g = np.random.default_rng(5)
T_seq, d_seq = 6, 4
E_seq = _g.normal(size=(T_seq, d_seq))
Wq, Wk, Wv = (_g.normal(size=(d_seq, d_seq)) for _ in range(3))

def causal_attention(E, Wq, Wk, Wv):
    return None

att_out = causal_attention(E_seq, Wq, Wk, Wv)

def _ref_att(E):   # position by position, only looking at the prefix
    Q, K, V = E @ Wq, E @ Wk, E @ Wv
    return np.array([softmax(Q[t] @ K[:t + 1].T / np.sqrt(d_seq)) @ V[:t + 1] for t in range(len(E))])
check('attention output', att_out, _ref_att(E_seq))
if att_out is not None:
    _E2 = E_seq.copy(); _E2[-1] += 10
    check('earlier positions unchanged when the last item changes', causal_attention(_E2, Wq, Wk, Wv)[:-1], att_out[:-1])

<details>
<summary><b>💡 Hint</b></summary>

`S = Q @ K.T / np.sqrt(d)`; `S[np.triu_indices(T, 1)] = -np.inf`; `softmax(S, axis=1) @ V`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The mask makes output $t$ a weighted average of values at positions $\le t$ only. SASRec reads off the prediction for item $t+1$ as
$\langle \text{out}_t, e_{i}\rangle$ for every candidate $i$. The real model adds positional embeddings, residual connections, layer norm and a feed-forward block ([[Transformers]]).

```python
_g = np.random.default_rng(5)
T_seq, d_seq = 6, 4
E_seq = _g.normal(size=(T_seq, d_seq))
Wq, Wk, Wv = (_g.normal(size=(d_seq, d_seq)) for _ in range(3))

def causal_attention(E, Wq, Wk, Wv):
    Q, K, V = E @ Wq, E @ Wk, E @ Wv
    S = Q @ K.T / np.sqrt(E.shape[1])
    S[np.triu_indices(len(E), 1)] = -np.inf
    return softmax(S, axis=1) @ V

att_out = causal_attention(E_seq, Wq, Wk, Wv)
```

</details>

### Exercise 17 · EASE: a closed-form linear autoencoder
*💻 Code · ★★★*

EASE (Steck 2019) learns an item–item matrix $B$ with **zero diagonal** that reconstructs the interaction matrix: $\min_B\lVert X - XB\rVert_F^2 + \lambda\lVert B\rVert_F^2$ s.t. $\mathrm{diag}(B)=0$.
The closed form is $P = (X^\top X + \lambda I)^{-1}$, $B_{ij} = -P_{ij}/P_{jj}$ ($i\neq j$), $B_{jj}=0$.

1. Implement `ease(X, lam)`.
2. Verify that column $j$ of $B$ equals a ridge regression of $X_{:,j}$ on the **other** columns.
3. Compute leave-one-out recall@10 on `X_ease_tr` / `ease_test` ($\lambda=10$) for EASE (`recall_ease`) and popularity (`recall_pop`).

In [ ]:
def ease(X, lam):
    return None

def loo_recall(S, K=10):
    # TODO: mask training items, fraction of users (with a test item) whose test item is in the top-K
    return None

recall_ease = None
recall_pop = None

_B = ease(X_ease_tr, 10.0)
if _B is not None:
    for j in [0, 7, 30]:
        _others = [c for c in range(60) if c != j]
        _w = Ridge(alpha=10.0, fit_intercept=False).fit(X_ease_tr[:, _others], X_ease_tr[:, j]).coef_
        check(f'column {j} = ridge on the other items', _B[_others, j], _w)
    check('zero diagonal', np.abs(np.diag(_B)).max(), 0)
    check('EASE beats popularity', recall_ease > recall_pop + 0.2, True)
else:
    check('EASE', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

`P = np.linalg.inv(X.T @ X + lam * np.eye(n))`; `B = -P / np.diag(P)` divides column $j$ by $P_{jj}$; then `np.fill_diagonal(B, 0)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Scores for all users are `X @ B`. Set already-seen items to `-inf` before taking the top-K.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The zero-diagonal constraint stops the trivial solution $B=I$ (copy the input). Each item is then predicted from the *other* items, like item-kNN but with
**learned** weights (including negative ones). Here EASE reaches recall@10 ≈ 0.82 vs ≈ 0.45 for popularity. This is exactly the "strong baseline"
point of the note: one matrix inversion often rivals deep autoencoders (Mult-VAE) on real benchmarks.

```python
def ease(X, lam):
    n = X.shape[1]
    P = np.linalg.inv(X.T @ X + lam * np.eye(n))
    B = -P / np.diag(P)
    np.fill_diagonal(B, 0.0)
    return B

def loo_recall(S, K=10):
    S = S.astype(float).copy()
    S[X_ease_tr > 0] = -np.inf
    top = np.argsort(-S, axis=1)[:, :K]
    return float(np.mean([ease_test[u] in top[u] for u in range(len(S)) if ease_test[u] >= 0]))

recall_ease = loo_recall(X_ease_tr @ ease(X_ease_tr, 10.0))
recall_pop = loo_recall(np.tile(X_ease_tr.sum(0), (len(X_ease_tr), 1)))
print('EASE', round(recall_ease, 3), '| popularity', round(recall_pop, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Deep Learning Recommenders](Deep%20Learning%20Recommenders.md).*